# Image-based GT rotation diagnostic — read-only

This notebook compares four fixed hypotheses against relative rotations estimated from images. Original source IDs 0 through 5 are excluded from evaluation because they belong to the initial calibration samples. Groups with a common recorded camera centre are selected from the beginning, middle, and end of the sequence.

**This experiment checks relative rotation only. It does not establish translation accuracy, capture timing, or absolute calibration.** Poor image matching makes a result inconclusive; it does not demonstrate a GT error.

Before running, install the dependencies in the notebook's Python environment: `python -m pip install -r requirements-diagnostics.txt`

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
SEQUENCE = ROOT / 'standard_GT' / 'city01_route01'
# Original, headerless capture CSV, NOT the exported poses.csv:
SOURCE_CSV = ROOT / 'dataset' / 'positions_angles_camera1.csv'
GROUPS_PER_REGION = 2
SEED = 17
assert SEQUENCE.is_dir(), f'Edit SEQUENCE: {SEQUENCE}'
assert SOURCE_CSV.is_file(), f'Edit SOURCE_CSV: {SOURCE_CSV}'


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from diagnostics.rotation_check import load_inputs, choose_groups, run_diagnostic, save_report, HYPOTHESES
ds, source, rows = load_inputs(SEQUENCE, SOURCE_CSV)
groups = choose_groups(ds, source, rows, GROUPS_PER_REGION)
selection = [{'region': g['region'], 'exported_frames': g['frames'],
              'source_ids': [int(ds.rows[k]['source_frame_id']) for k in g['frames']],
              'csv_rows_zero_based': rows[g['frames']].tolist()} for g in groups]
display(pd.DataFrame(selection))
missing = {'beginning', 'middle', 'end'} - {g['region'] for g in groups}
print('Missing regions:', sorted(missing))
assert groups, 'No eligible groups. Do not replace these with moving pairs without checking parallax.'


### Run the diagnostic

For each image pair, the diagnostic matches SIFT features and robustly estimates a rotation. It holds out 30% of the matches from fitting to check consistency independently. When sufficient matches are available, it also checks agreement between the two panorama halves.

- `current_gt`: the current exported GT.
- `offset_only`: the previous row in the original capture CSV.
- `axes_only`: the proposed camera-axis convention.
- `offset_and_axes`: both changes together.

The row offset applies to the original CSV, not to the list of retained images.

In [ ]:
report = run_diagnostic(SEQUENCE, SOURCE_CSV, GROUPS_PER_REGION, SEED)
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUT = ROOT / 'outputs' / 'gt_rotation_diagnostic' / SEQUENCE.name / stamp
save_report(report, OUT)
print('Report saved:', OUT)


In [ ]:
df = pd.DataFrame(report['results'])
error_columns = [h + '_error_deg' for h in HYPOTHESES]
for col in error_columns:
    if col not in df: df[col] = float('nan')
view = ['region', 'group', 'source_id_i', 'source_id_j', 'status',
        'heldout_inlier_fraction', 'hemisphere_disagreement_deg'] + error_columns
display(df.reindex(columns=view))
usable = df.loc[df['status'].eq('usable')].copy()
print(f'Usable pairs: {len(usable)}/{len(df)}; independent sampled groups: {usable.group.nunique()}')
print('Regions without selected groups:', report['regions_missing'])
if len(usable):
    summary = usable.groupby('region')[error_columns].agg(['count', 'median', 'max'])
    display(summary)
    summary.to_csv(OUT / 'summary.csv')
else:
    print('INCONCLUSIVE: no pair passed image consistency checks.')


In [ ]:
if len(usable):
    fig, ax = plt.subplots(figsize=(12, 5))
    labels = [f"{r.region}: {r.source_id_i}→{r.source_id_j}" for r in usable.itertuples()]
    for h in HYPOTHESES:
        ax.plot(range(len(usable)), usable[h + '_error_deg'], 'o-', label=h)
    ax.set_xticks(range(len(usable)), labels, rotation=65, ha='right')
    ax.set_ylabel('Relative rotation geodesic error (degrees)')
    ax.set_title('Only pairs passing image consistency checks')
    ax.legend(); ax.grid(alpha=.25); fig.tight_layout()
    fig.savefig(OUT / 'rotation_errors.png', dpi=160)
    plt.show()
import shutil
archive = shutil.make_archive(str(OUT.parent / (OUT.name + '_report')), 'zip', OUT)
print('Send this ZIP for interpretation:', archive)


### Interpret the results

- Lower error for the combined hypothesis across several independent groups supports an axis-convention/rotation-label mismatch. It does not prove that the entire dataset is correct.
- A disagreement greater than 1 degree between panorama halves makes the result inconclusive. A missing value means that at least one half lacked a reliable fit; it does not confirm agreement between the halves.
- The three pairs within a group are not independent. Consider the number of groups and their coverage along the sequence.
- A common camera centre is checked using metadata. The actual capture centres have not been independently verified.
- This notebook does not apply any GT correction. The ZIP contains diagnostic reports and, when usable fits are available, a plot. It does not contain the original images.
